In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [66]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()

        pos_enc = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        args = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pos_enc[:, 0::2] = torch.sin(position * args)
        pos_enc[:, 1::2] = torch.cos(position * args)
        pos_enc = pos_enc.unsqueeze(0)

        self.register_buffer('pe', pos_enc)

    def forward(self, x):
        len_ = x.size(1)
        return self.pe[:, :len_, :].to(x.device)


In [67]:
def subsequent_mask(size):
    return torch.tril(torch.ones((1, size, size), device=device, dtype=torch.bool))

def make_pad_mask(seq, pad_idx):
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


In [68]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, d_head, dropout=0.3):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.d_head = d_head
        self.dropout = nn.Dropout(dropout)
        self.inner_dim = n_head * d_head

        self.w_q = nn.Linear(d_model, self.inner_dim, bias=False)
        self.w_k = nn.Linear(d_model, self.inner_dim, bias=False)
        self.w_v = nn.Linear(d_model, self.inner_dim, bias=False)
        self.w_out = nn.Linear(self.inner_dim, d_model, bias=False)


    def forward(self, q, k, v, mask=None):
        B = q.size(0)

        Q = self.w_q(q).view(B, -1, self.n_head, self.d_head).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_head, self.d_head).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_head, self.d_head).transpose(1, 2)

        attention_score = Q @ K.transpose(-2, -1) / self.d_head ** 0.5

        if mask is not None:
            attention_score = attention_score.masked_fill(~mask, float('-inf'))

        attention_score = F.softmax(attention_score, dim=-1)
        attention_score = self.dropout(attention_score) # можно убрать

        out = attention_score @ V

        out = out.transpose(1, 2).reshape(B, -1, self.inner_dim)

        out = self.w_out(out)
        out = self.dropout(out)

        return out


In [69]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [70]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_head, d_ff, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_head, d_head, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, seq_mask=None):
        attention_res = self.attention(x, x, x, mask=seq_mask)
        x = x + self.dropout(attention_res)
        x = self.norm1(x)

        ff = self.ff(x)
        x = x + self.dropout(ff)
        x = self.norm2(x)
        
        return x

In [71]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_head, d_ff, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, d_head, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_head, d_head, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, seq_mask=None, target_mask=None):
        self_attention_res = self.self_attention(x, x, x, mask=target_mask)
        x = x + self.dropout(self_attention_res)
        x = self.norm1(x)
        
        cross_attention_res = self.cross_attention(x, enc_output, enc_output, mask=seq_mask)
        x = x + self.dropout(cross_attention_res)
        x = self.norm2(x)
        
        ff = self.ff(x)
        x = x + self.dropout(ff)
        x = self.norm3(x)

        return x

In [72]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = SinusoidalPositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, n_head, d_head, d_ff, dropout) for _ in range(n_layer)]
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, seq, seq_mask=None):
        x = self.token_embed(seq)
        x = x + self.pos_embed(x)
        
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, seq_mask=seq_mask)
        x = self.norm(x)
        
        return x

In [73]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = SinusoidalPositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, n_head, d_head, d_ff, dropout) for _ in range(n_layer)]
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, target, encoder_output, seq_mask=None, target_mask=None):
        x = self.token_embed(target)
        x = x + self.pos_embed(x)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, encoder_output, seq_mask=seq_mask, target_mask=target_mask)
        x = self.norm(x)
        
        return x

In [74]:
class Transformer(nn.Module):
    def __init__(self, vocab_size_seq, vocab_size_target, d_model=512, n_layer=6, n_head=8, d_head=64, d_ff=2048, max_len=512, dropout=0.3, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(vocab_size_seq, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout)
        self.decoder = Decoder(vocab_size_target, d_model, n_layer, n_head, d_head, d_ff, max_len, dropout)
        self.out = nn.Linear(d_model, vocab_size_target)
        self.pad_idx = pad_idx

    def forward(self, seq, target):
        seq_mask = make_pad_mask(seq, self.pad_idx)
        target_pad_mask = make_pad_mask(target, self.pad_idx)
        
        target_len = target.size(1)
        causal_mask = subsequent_mask(target_len).unsqueeze(0)
        target_mask = target_pad_mask & causal_mask
        
        encoder_output = self.encoder(seq, seq_mask=seq_mask)
        decoder_output = self.decoder(target, encoder_output, seq_mask=seq_mask, target_mask=target_mask)
        logits = self.out(decoder_output)

        return logits

In [76]:
vocab_seq = vocab_target = 28
d_model = 128
n_layer = 2
n_head = 4
d_head = 32
d_ff = 512
max_len = 64

model = Transformer(
    vocab_size_seq=vocab_seq, 
    vocab_size_target=vocab_target, 
    d_model=d_model, 
    n_layer=n_layer, 
    n_head=n_head, 
    d_head=d_head,
    d_ff=d_ff, 
    max_len=max_len, 
    dropout=0.3
).to(device)

B = 4
T_seq = 10
T_target = 12
seq = torch.randint(1, vocab_seq, (B, T_seq), device=device)
target = torch.randint(1, vocab_target, (B, T_target), device=device)

logits = model(seq, target[:, :-1])
labels = target[:, 1:]

loss = F.cross_entropy(logits.reshape(-1, vocab_target), labels.reshape(-1))
print(loss.item())

3.5119524002075195


# Tokenizer

In [78]:
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [79]:
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

In [81]:
class Tokenizer:
    def __init__(self):
        self.word2idx = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
        self.idx2word = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN, 3: UNK_TOKEN}
        self.vocab_size = 4
    
    def fit(self, sentences, max_words=10000):
        """Собирает словарь из списка предложений"""
        counter = Counter()
        for sent in sentences:
            tokens = self._tokenize(sent)
            counter.update(tokens)
        
        most_common = counter.most_common(max_words)
        for word, _ in most_common:
            if word not in self.word2idx:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1
                
    def _tokenize(self, text):
        text = text.lower().strip()
        text = re.sub(r"[^\w\s]", "", text)
        return text.split()

    def encode(self, text, add_special_tokens=True):
        tokens = self._tokenize(text)
        indices = [self.word2idx.get(t, UNK_IDX) for t in tokens]
        if add_special_tokens:
            return [SOS_IDX] + indices + [EOS_IDX]
        return indices

    def decode(self, indices):
        tokens = []
        for idx in indices:
            idx = idx.item() if isinstance(idx, torch.Tensor) else idx
            if idx == EOS_IDX: break
            if idx != PAD_IDX and idx != SOS_IDX:
                tokens.append(self.idx2word.get(idx, UNK_TOKEN))
                
        return " ".join(tokens)


# Dataloader

In [82]:
def load_data(filepath, limit=None):
    """Читает файл и возвращает пары (src, tgt)"""
    pairs = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                src = parts[1]
                tgt = parts[3]
                pairs.append((src, tgt))
                
    return pairs


# Dataset

In [83]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_tokenizer, tgt_tokenizer):
        self.pairs = pairs
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src_idxs = self.src_tokenizer.encode(src_text)
        tgt_idxs = self.tgt_tokenizer.encode(tgt_text)
        
        return torch.tensor(src_idxs), torch.tensor(tgt_idxs)

In [84]:
def collate_fn(batch):
    """Функция для DataLoader, чтобы выровнять длину предложений (padding)"""
    src_batch, tgt_batch = zip(*batch)
    
    src_padded = torch.nn.utils.rnn.pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_padded = torch.nn.utils.rnn.pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_padded, tgt_padded


In [85]:
DATA_PATH = "data/RusEngTranslate.tsv"

raw_data = load_data(DATA_PATH, limit=100) 
src_sents = [p[0] for p in raw_data]
tgt_sents = [p[1] for p in raw_data]

src_tokenizer = Tokenizer()
tgt_tokenizer = Tokenizer()
src_tokenizer.fit(src_sents)
tgt_tokenizer.fit(tgt_sents)

print(f"Vocab Source: {src_tokenizer.vocab_size},"
      f"Vocab Target: {tgt_tokenizer.vocab_size}")

dataset = TranslationDataset(raw_data, src_tokenizer, tgt_tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)


Vocab Source: 218,Vocab Target: 260


# Обучение

In [86]:
model = Transformer(
    vocab_size_seq=src_tokenizer.vocab_size,
    vocab_size_target=tgt_tokenizer.vocab_size,
    d_model=128,
    n_layer=2,
    n_head=4,
    d_head=32,
    d_ff=256,
    max_len=100,
    dropout=0.1,
    pad_idx=PAD_IDX
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [87]:
model.train()
num_epochs = 50

for epoch in range(num_epochs):
    total_loss = 0
    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_y = tgt[:, 1:]
        
        optimizer.zero_grad()
        
        logits = model(src, tgt_input)
        
        loss = criterion(logits.reshape(-1, tgt_tokenizer.vocab_size), tgt_y.reshape(-1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")


Epoch 10, Loss: 1.0597
Epoch 20, Loss: 0.3035
Epoch 30, Loss: 0.2266
Epoch 40, Loss: 0.2227
Epoch 50, Loss: 0.2106


# Inference

In [90]:
model.eval()
test_sentence = "I will be back soon"
src_idxs = torch.tensor(src_tokenizer.encode(test_sentence)).unsqueeze(0).to(device)

generated = torch.tensor([[SOS_IDX]]).to(device)

with torch.no_grad():
    for _ in range(10):
        logits = model(src_idxs, generated)
        next_token_logits = logits[:, -1, :]
        next_token = next_token_logits.argmax(dim=-1).unsqueeze(1)
        
        generated = torch.cat([generated, next_token], dim=1)
        
        if next_token.item() == EOS_IDX:
            break

decoded_translation = tgt_tokenizer.decode(generated[0])
print(f"Input: {test_sentence}")
print(f"Output: {decoded_translation}")

Input: I will be back soon
Output: скоро вернусь
